## Setup Decoder

In [2]:
from can_decoder.decoder import Decoder

## Import Data, Setup and Signal Decoding

In [ ]:
#TODO: Direct reading from Vehicle Spy files 

## Read csv
decoder = Decoder(csv_file_path = "./data/SampleRoadAcceleration_v2.csv")

## Add byte filtering
byte_filters = {
    "18DA10F1": ["1", "2", "3", "4",'5'],
    "18DAF110": ["1", "2", "3", "4",'5'],
    "18DAF218": ["1", "2", "3", "4",'5'],
    "18DA18F2": ["1", "2", "3", "4",'5'],
}

## Message Generation
decoder.generate_msgs(byte_filters=byte_filters)

## Dynamic filtering and corrections: 
# 1) If byte 1 is 0 we should discard byte 5
for msg in decoder.msgs:
    if msg.msg_byte_filter is not None:
        if msg.msg_byte_filter['1'] != 0:
            msg.msg_byte_filter.pop('5', None)
            # Remove duplicates

# 2) Remove duplicate messages with same msg_id and identical msg_byte_filter after discarding byte 5
unique_msgs = []
seen = set()
for msg in decoder.msgs:
    key = (msg.msg_id, tuple(sorted(msg.msg_byte_filter.items())) if msg.msg_byte_filter else None)
    if key not in seen:
        unique_msgs.append(msg)
        seen.add(key)
decoder.msgs = unique_msgs





In [7]:
## Generate message time series data
decoder.generate_msg_ts_data(rewrite=True) 

In [8]:
## Calculate signals
decoder.calculate_signals(
        tokenization_method='conditional_bit_flip',
        signedness_method='msb_classifier',
        alpha1=0.01,
        alpha2=0.5, #0.5,
        gamma1=0.2)

## Visualization

In [14]:
## Plot message from id, to see what signals have been decoded
decoder.plot_message_from_id('201')

# Have to specify byte filter if needed
decoder.plot_message_from_id('18DAF218',[0,15,98,0x50,0x22])


In [ ]:
## Signal Visualization
f = decoder.plot_signal_from_id('18DAF218',[0,15,98,0x50,0x22], signal_id='S_18DAF218_BE_3', return_fig=False)

# If return_fig=True, it returns the figure object which can be further customized
# f = decoder.plot_signal_from_id('18DAF218',[0,15,98,0x50,0x22], signal_id='S_18DAF218_BE_3', return_fig=True)
# f['data'][0]['mode'] = 'lines+markers'
# f['data'][0]['marker'] = {'size': 5, 'color': 'red'}
# f.show()



## Signal Matching (Unknown to Known)

In [22]:
## Simple Signal comparison

# signal_a is the known signal and signal_b is the candidate signal we want to compare with signal_a.
signal_a = decoder.get_signal('18DAF218',[0,15,98,0x50,0x22], 'S_18DAF218_BE_3')
signal_b = decoder.get_signal('201',signal_id='S_201_BE_3')

#  The simple function to compare two signals is signal_match():
result = decoder.signal_match(signal_a, signal_b,)
result

(LinregressResult(slope=np.float64(0.1250021459218392), intercept=np.float64(-0.5357254467001695), rvalue=np.float64(0.9998628518261544), pvalue=np.float64(0.0), stderr=np.float64(0.0001193407901337974), intercept_stderr=np.float64(0.0979450869034786)),
 np.float64(0.9997257224619305))

In [ ]:
## Full sweep:
# The function find_signal_match() looks through all signals, compares them (linear regression) and keeps the ones whose R^2 is above a certain threshold (e.g. 0.5). 
# It returns a list of candidate matches for each reference signal
# CAREFUL: Computationally expensive/might take time
candidates_speed = decoder.find_signal_match(
    signal = decoder.get_signal('18DAF218',[0,15,98,0x50,0x22], 'S_18DAF218_BE_3'),
    thresh=0.9
)

S_18DAF218_BE_3      --> S_208_BE_9           in 208        , r^2=0.999770
S_18DAF218_BE_3      --> S_208_LE_13          in 208        , r^2=0.989309
S_18DAF218_BE_3      --> S_102_BE_1           in 102        , r^2=0.999771
S_18DAF218_BE_3      --> S_102_BE_19          in 102        , r^2=0.999931
S_18DAF218_BE_3      --> S_102_BE_21          in 102        , r^2=0.999563
S_18DAF218_BE_3      --> S_102_LE_20          in 102        , r^2=0.997801
S_18DAF218_BE_3      --> S_102_LE_23          in 102        , r^2=0.998039
S_18DAF218_BE_3      --> S_102_LE_53          in 102        , r^2=0.975616
S_18DAF218_BE_3      --> S_103_BE_2           in 103        , r^2=0.999788
S_18DAF218_BE_3      --> S_103_BE_5           in 103        , r^2=0.999932
S_18DAF218_BE_3      --> S_103_BE_7           in 103        , r^2=0.999564
S_18DAF218_BE_3      --> S_103_BE_9           in 103        , r^2=0.999462
S_18DAF218_BE_3      --> S_103_LE_15          in 103        , r^2=0.974462
S_18DAF218_BE_3      --> 

In [ ]:
## Plot matches for comparison (normalized)
# First signal is the reference (black, with markers)
f = decoder.plot_signal_matches(known_signal_speed, candidates_speed, return_fig=False)

## TO-DOs

In [ ]:
# TODO: Show full usage of all decoder functions (like get_signal, get_message etc)
# TODO: Add widgets on decoding parameters (done at the end of jeep_test)
